In [1]:
# DO NOT CONTAINERISE
# =====
# Dependency
# -----
# ! pip install -r requirements.txt
# ! pip list
# ! conda list

# !conda install -y requests

import os, glob
from datetime import datetime

import re
import time
import math

import urllib.parse
import requests

import pandas as pd

from tqdm import tqdm  # status bars

date_time_now = datetime.now().strftime('%Y%m%d_%H%M%S%f')

# base settings
# -----
conf_vlab_name     = "NIS"
# conf_workflow_name = "Marine"

# conf_workflow_id   = f"wid-{datetime.now().strftime('%Y%m%d_%H%M%S%f')}"
param_workflow_name = "workflow name"

# dev
# -----
# library: --volume="//c/DockerShare/DNA:/home/jovyan" naavre-fl-dna-jupyter:local
# NaaVRE: /home/jovyan/Virtual Labs/DNA/Git public
# conf_dir_code = os.path.join("/", "home", "jovyan", "Virtual Labs", conf_vlab_name, "Git public", "library")
# if not os.path.exists(conf_dir_code):
#     os.makedirs(conf_dir_code)

# conf_dir_data  = os.path.join("/", "home", "jovyan", "Cloud Storage", "naa-vre-user-data", conf_vlab_name, param_workflow_name)
# if not os.path.exists(conf_dir_data):
#     os.makedirs(conf_dir_data)

# local
# -----
conf_dir_workspace = os.path.join("/", "home", "jovyan", "Cloud Storage")

conf_dir_data_local_tmp = os.path.join("/", "tmp", "data")

# MINIO
# -----
conf_minio_public_bucket      = "naa-vre-public"
conf_minio_public_bucket_root = f"vl-{conf_vlab_name.lower()}"
conf_minio_public_local_root  = os.path.join(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root)
conf_minio_public_local_code  = os.path.join(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root, "code")
conf_minio_public_local_data  = os.path.join(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root, "data")

conf_minio_user_bucket        = "naa-vre-user-data"
# conf_minio_user_bucket_root   = param_user_email
conf_minio_user_bucket_root   = conf_vlab_name
conf_minio_user_local_root    = os.path.join(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root)
conf_minio_user_local_code    = os.path.join(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root,   "library")
conf_minio_user_local_data    = os.path.join(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root,   param_workflow_name)
conf_minio_user_local_flog    = os.path.join(conf_minio_user_local_data, "log.md")

# for workflow step
# .....
# if os.path.exists(conf_minio_user_local_flog):
#     with open(conf_minio_user_local_flog, "a+") as fp_log:
#         fp_log.write(f"\n## {workflow_step}\n") 
# else:
#     if not os.path.exists(conf_minio_user_local_data):
#         os.makedirs(conf_minio_user_local_data)
#     with open(conf_minio_user_local_flog, "w+") as fp_log:
#         fp_log.write(f"\n## {workflow_step}\n") 

# API key
# -----
# If running under NaaVRE, input `your api key` with the correct value and input in the GUI:
# secret_SERVICE_KEY = "d18e08911c964d45912eb1e954adf994"
# secret_SERVICE_KEY = SecretsProvider().set_secret("secret_SERVICE_KEY")
# secret_SERVICE_KEY = SecretsProvider().get_secret("secret_SERVICE_KEY")

# Input param
# -----
BASE_OBIS           = "https://api.obis.org/v3"
BASE_MARINE_REGIONS = "https://www.marineregions.org/rest"
BASE_WORMS          = "https://www.marinespecies.org/rest"

# USER file paths & parameters
# .....
# Input
fname_locations      = "input-locations.tsv"  # your uploaded locations file
param_file_locations = os.path.join(conf_minio_user_local_root, fname_locations)

fname_species        = "input-species.tsv"  # your uploaded species file
param_file_species   = os.path.join(conf_minio_user_local_root, fname_species)

# Output
fname_result         = "output-results.tsv"
param_file_output    = os.path.join(conf_minio_user_local_data, fname_result)

# Parameters
param_MAX_OBIS_RECORDS       = 10000  # how many OBIS occurrences to consider
param_MAX_REGION_DIAGONAL_KM = 3000   # filter out marine regions larger than this

# result
# .....
results = []

print("Finish: NaaVRE parameters")
print(f"Workspace public:")
print(f"  Root: {conf_minio_public_local_root}")
print(f"  Code: {conf_minio_public_local_code}")
print(f"  Data: {conf_minio_public_local_data}")

print(f"Workspace user:")
print(f"  Root: {conf_minio_user_local_root}")
print(f"  Code: {conf_minio_user_local_code}")
print(f"  Data: {conf_minio_user_local_data}")
print(f"  Log:  {conf_minio_user_local_flog}")


Finish: NaaVRE parameters
Workspace public:
  Root: /home/jovyan/Cloud Storage/naa-vre-public/vl-nis
  Code: /home/jovyan/Cloud Storage/naa-vre-public/vl-nis/code
  Data: /home/jovyan/Cloud Storage/naa-vre-public/vl-nis/data
Workspace user:
  Root: /home/jovyan/Cloud Storage/naa-vre-user-data/NIS
  Code: /home/jovyan/Cloud Storage/naa-vre-user-data/NIS/library
  Data: /home/jovyan/Cloud Storage/naa-vre-user-data/NIS/workflow name
  Log:  /home/jovyan/Cloud Storage/naa-vre-user-data/NIS/workflow name/log.md


In [2]:
# DNA, workflow start
# ---
# NaaVRE:
#  cell:
#   outputs:
#    - dummy_cell_arg_o: String
# ...

import os
import sys
from datetime import datetime

# prepare folders
# .....
if not os.path.exists(conf_dir_data_local_tmp):
    os.makedirs(conf_dir_data_local_tmp)

# if not os.path.exists(conf_minio_public_local_root):
#     os.makedirs(conf_minio_public_local_root)

if not os.path.exists(conf_minio_user_local_root):
    os.makedirs(conf_minio_user_local_root)

if not os.path.exists(conf_minio_user_local_data):
    os.makedirs(conf_minio_user_local_data)
    
with open(conf_minio_user_local_flog, "w+") as fp_log:
    fp_log.write(f"# {param_workflow_name}\n")

# create log
# .....
print(param_workflow_name)
workflow_step = f"{conf_vlab_name}-Start"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n")
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n")

# lib, minio_public
# -----
# sys.path.append(conf_minio_public_local_code)

# lib, minio_user
# -----
# sys.path.append(conf_minio_user_local_code)

# input
# -----
dummy_cell_arg_i = "dummy input"

# output
# -----
dummy_cell_arg_o = "dummy output"

# func
# -----

# start
# -----

# finish
# -----
with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")
    fp_log.write(f"\nOutput: {conf_minio_user_local_data}\n")

print(f"Finish: {workflow_step}")


workflow name
Finish: NIS-Start


In [3]:
# NIS, Read input data
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - dummy_cell_arg_i: String
#   outputs:
#    - dummy_cell_arg_o: String
# ...

import os
import sys
from datetime import datetime

print(param_workflow_name)
workflow_step = f"{conf_vlab_name}-Read input data"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 

# lib, minio_public
# -----
# sys.path.append(conf_minio_public_local_code)

# lib, minio_user
# -----
# sys.path.append(conf_minio_user_local_code)

# input
# -----
dummy_cell_arg_i = "dummy input"

# output
# -----
dummy_cell_arg_o = "dummy output"

# func
# -----
# dms_to_decimal()

# 0. Small helpers
# .....
def dms_to_decimal(dms_str):
    """
    Convert a coordinate in DMS format like '51°06'39.3"N'
    or '2°36'13.6"E' to decimal degrees.
    """
    if not isinstance(dms_str, str):
        raise ValueError(f"Invalid DMS value: {dms_str}")

    # matches: degrees, minutes, seconds, hemisphere (N/S/E/W)
    m = re.match(
        r"^\s*(\d+)[°:\s]+(\d+)[\'’:\s]+(\d+(?:\.\d+)?)[\"”]?\s*([NSEW])\s*$",
        dms_str.strip()
    )
    if not m:
        raise ValueError(f"Cannot parse DMS: {dms_str}")

    deg, mins, secs, hemi = m.groups()
    deg = float(deg)
    mins = float(mins)
    secs = float(secs)

    decimal = deg + mins / 60.0 + secs / 3600.0
    if hemi in ("S", "W"):
        decimal = -decimal
    return decimal


# Main-1 Read input TSVs
# .....
print(f"Reading locations from {param_file_locations}")
df_locations = pd.read_csv(param_file_locations, sep="\t")

print(f"Reading species from {param_file_species}")
species = pd.read_csv(param_file_species, sep="\t")

# Main-2 Convert DMS to decimal degrees
# .....
print("Converting coordinates to decimal degrees...")

df_locations["decimalLatitude"]  = df_locations["verbatimLatitude"].apply(dms_to_decimal)
df_locations["decimalLongitude"] = df_locations["verbatimLongitude"].apply(dms_to_decimal)

# Main-3 Join species with df_locations via locationID
# .....
df_merged = species.merge(df_locations, on="locationID", how="inner")
print(f"Total (location, species) cases to analyse: {len(df_merged)}")

# finish
# -----
with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")

print(f"Finish: {workflow_step}")


workflow name
Reading locations from /home/jovyan/Cloud Storage/naa-vre-user-data/NIS/input-locations.tsv
Reading species from /home/jovyan/Cloud Storage/naa-vre-user-data/NIS/input-species.tsv
Converting coordinates to decimal degrees...
Total (location, species) cases to analyse: 50
Finish: NIS-Read input data


In [4]:
# Find Closest Obis Occurrence
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - dummy_cell_arg_i: String
#   outputs:
#    - dummy_cell_arg_o: String
# ...

# func: find_closest_obis_occurrence()

import os
import sys
from datetime import datetime

print(param_workflow_name)
workflow_step = f"{conf_vlab_name}-Find Closest Obis Occurrence"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 

# lib, minio_public
# -----
# sys.path.append(conf_minio_public_local_code)

# lib, minio_user
# -----
# sys.path.append(conf_minio_user_local_code)

# input
# -----
dummy_cell_arg_i = "dummy input"

# output
# -----
dummy_cell_arg_o = "dummy output"

# func
# -----
# 1. OBIS helpers: occurrences + sea-only distance
# .....
def get_obis_occurrences(scientific_name, limit=1000):
    """
    Retrieve occurrence coordinates for a given species from OBIS.
    Returns a list of dicts with:
      - decimalLatitude
      - decimalLongitude
      - occurrenceID (OBIS occurrence id, if present)
    """
    url = f"{BASE_OBIS}/occurrence"
    params = {
        "scientificname": scientific_name,
        "size": limit
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()

    records = []
    for rec in data.get("results", []):
        lat = rec.get("decimalLatitude")
        lon = rec.get("decimalLongitude")
        if lat is None or lon is None:
            continue
        records.append({
            "decimalLatitude": float(lat),
            "decimalLongitude": float(lon),
            "occurrenceID": rec.get("id")
        })
    return records


def sea_distance_km(ref_lat, ref_lon, lat, lon):
    """
    Shortest sea route distance in km using searoute.

    Requires:
      pip install searoute
    """
    import searoute as sr  # imported here to avoid dependency if not needed

    origin = [ref_lon, ref_lat]     # searoute expects [lon, lat]
    destination = [lon, lat]

    route = sr.searoute(origin, destination, units="km")
    if route is None:
        return None
    return float(route.properties["length"])


# start
# -----
# def find_closest_obis_occurrence(scientific_name, new_lat, new_lon, limit=200):

# finish
# -----
with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")

print(f"Finish: {workflow_step}")


workflow name
Finish: NIS-Find Closest Obis Occurrence


In [5]:
# Classify with Progress
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - dummy_cell_arg_i: String
#   outputs:
#    - dummy_cell_arg_o: String
# ...

# func: classify_with_progress()

import os
import sys
from datetime import datetime

print(param_workflow_name)
workflow_step = f"{conf_vlab_name}-Classify with Progress"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 

# lib, minio_public
# -----
# sys.path.append(conf_minio_public_local_code)

# lib, minio_user
# -----
# sys.path.append(conf_minio_user_local_code)

# input
# -----
dummy_cell_arg_i = "dummy input"

# output
# -----
dummy_cell_arg_o = "dummy output"

# func
# -----
# 2. Marine Regions helpers (with bbox size filter)
# .....
# classify_with_progress(): get_filtered_mrgids_for_coord()

# cache for MRGID -> diagonal distance (km)
_region_diag_cache = {}


def bbox_diagonal_km_from_degrees(min_lat, min_lon, max_lat, max_lon):
    """
    Approximate diagonal distance (km) of a lat/lon bounding box
    using simple degree-to-km conversions.
    """
    if not (-90 <= min_lat <= 90 and -90 <= max_lat <= 90):
        return None
    if not (-180 <= min_lon <= 180 and -180 <= max_lon <= 180):
        return None

    dlat_deg = max_lat - min_lat
    dlon_deg = max_lon - min_lon

    # Normalize longitude difference to [-180, 180]
    if dlon_deg > 180:
        dlon_deg -= 360
    elif dlon_deg < -180:
        dlon_deg += 360

    lat_km_per_deg = 111.32
    mid_lat = (min_lat + max_lat) / 2.0
    lon_km_per_deg = 111.32 * math.cos(math.radians(mid_lat))

    dlat_km = dlat_deg * lat_km_per_deg
    dlon_km = dlon_deg * lon_km_per_deg

    diag_km = math.sqrt(dlat_km**2 + dlon_km**2)

    if diag_km > 25000:
        return None

    return diag_km


# .....
def get_region_bbox_diagonal_km(mrgid):
    """
    Get approximate diagonal (km) of the bounding box for a Marine Region (MRGID).
    Uses a small cache to avoid repeated calls for the same MRGID.
    """
    mrgid_str = str(mrgid)
    if mrgid_str in _region_diag_cache:
        return _region_diag_cache[mrgid_str]

    url = f"{BASE_MARINE_REGIONS}/getGazetteerRecordByMRGID.json/{mrgid_str}/"

    try:
        resp = requests.get(url, headers={"accept": "application/json"}, timeout=15)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"[WARN] Request error for MRGID {mrgid_str}: {e}")
        _region_diag_cache[mrgid_str] = None
        return None

    try:
        data = resp.json()
    except ValueError:
        print(f"[WARN] Could not decode JSON for MRGID {mrgid_str}")
        _region_diag_cache[mrgid_str] = None
        return None

    # API may return a list or a dict
    if isinstance(data, list):
        if not data:
            _region_diag_cache[mrgid_str] = None
            return None
        data = data[0]

    if not isinstance(data, dict):
        _region_diag_cache[mrgid_str] = None
        return None

    min_lat = data.get("minLatitude")
    min_lon = data.get("minLongitude")
    max_lat = data.get("maxLatitude")
    max_lon = data.get("maxLongitude")

    if None in (min_lat, min_lon, max_lat, max_lon):
        _region_diag_cache[mrgid_str] = None
        return None

    try:
        min_lat = float(min_lat)
        min_lon = float(min_lon)
        max_lat = float(max_lat)
        max_lon = float(max_lon)
    except ValueError:
        _region_diag_cache[mrgid_str] = None
        return None

    diag_km = bbox_diagonal_km_from_degrees(min_lat, min_lon, max_lat, max_lon)
    _region_diag_cache[mrgid_str] = diag_km
    return diag_km

# classify_with_progress()
# .....
def get_filtered_mrgids_for_coord(lat_dd, lon_dd, max_diagonal_km=3000):
    """
    Get Marine Regions MRGIDs for a point and filter out regions whose bounding
    box diagonal is more than max_diagonal_km.

    Returns a list of MRGID strings that:
      - have bounding box info, AND
      - diagonal distance <= max_diagonal_km
    """
    try:
        lat = float(lat_dd)
        lon = float(lon_dd)
    except (TypeError, ValueError):
        return []

    url = f"{BASE_MARINE_REGIONS}/getGazetteerRecordsByLatLong.json/{lat}/{lon}/"

    try:
        resp = requests.get(url, headers={"accept": "application/json"}, timeout=15)
        resp.raise_for_status()
        regions = resp.json()
    except Exception as e:
        print(f"[WARN] Marine Regions request failed for {lat}, {lon}: {e}")
        return []

    if not regions or not isinstance(regions, list):
        return []

    filtered_mrgids = []

    for item in regions:
        mrgid = item.get("MRGID")
        if not mrgid:
            continue

        diag_km = get_region_bbox_diagonal_km(mrgid)

        # Keep only if bbox is known AND size <= threshold
        if diag_km is not None and diag_km <= max_diagonal_km:
            filtered_mrgids.append(str(mrgid))

    return filtered_mrgids


# 3. WoRMS/WRiMS helpers
# .....
# main(): classify_with_progress()

def get_aphia_id(scientific_name, marine_only=True):
    """
    Use WoRMS REST:
      GET /AphiaIDByName/{ScientificName}?marine_only=true

    Returns AphiaID (int) or None
    """
    _aphia_cache = {}
    
    if scientific_name in _aphia_cache:
        return _aphia_cache[scientific_name]

    encoded_name = urllib.parse.quote(scientific_name)
    url = f"{BASE_WORMS}/AphiaIDByName/{encoded_name}"

    params = {}
    if marine_only:
        params["marine_only"] = "true"

    try:
        r = requests.get(url, params=params, timeout=20)
        if r.status_code == 204 or not r.text.strip():
            _aphia_cache[scientific_name] = None
            return None
        r.raise_for_status()
        text = r.text.strip()
        if text == "0":
            _aphia_cache[scientific_name] = None
            return None
        aphia_id = int(text)
    except Exception as e:
        print(f"[WARN] Could not get AphiaID for '{scientific_name}': {e}")
        _aphia_cache[scientific_name] = None
        return None

    _aphia_cache[scientific_name] = aphia_id
    time.sleep(0.1)
    return aphia_id


def get_distributions_for_aphia(aphia_id):
    """
    Use WoRMS REST:
      GET /AphiaDistributionsByAphiaID/{ID}

    Returns a list of dicts.
    """
    _dist_cache  = {}
    
    if aphia_id in _dist_cache:
        return _dist_cache[aphia_id]

    url = f"{BASE_WORMS}/AphiaDistributionsByAphiaID/{aphia_id}"

    try:
        r = requests.get(url, timeout=20)
        if r.status_code == 204 or not r.text.strip():
            _dist_cache[aphia_id] = []
            return []
        r.raise_for_status()
        data = r.json()
        if not isinstance(data, list):
            data = []
    except Exception as e:
        print(f"[WARN] Could not get distributions for AphiaID {aphia_id}: {e}")
        _dist_cache[aphia_id] = []
        return []

    _dist_cache[aphia_id] = data
    time.sleep(0.1)
    return data


def extract_mrgid_from_locationID(locationID):
    """
    distribution['locationID'] looks like:
      'http://marineregions.org/mrgid/7130'
    We want the numeric MRGID as string, e.g. '7130'.
    """
    if not isinstance(locationID, str):
        return None
    m = re.search(r"/mrgid/(\d+)", locationID)
    return m.group(1) if m else None


# .....
def classify_invasiveness(scientific_name, mrgids_for_location):
    """
    Returns dict with:
      wrims_has_record, wrims_invasive, wrims_establishment, wrims_matching_mrgid
    """
    if not mrgids_for_location:
        return {
            "wrims_has_record": False,
            "wrims_invasive": None,
            "wrims_establishment": None,
            "wrims_matching_mrgid": None
        }

    aphia_id = get_aphia_id(scientific_name)
    if aphia_id is None:
        return {
            "wrims_has_record": False,
            "wrims_invasive": None,
            "wrims_establishment": None,
            "wrims_matching_mrgid": None
        }

    distribs = get_distributions_for_aphia(aphia_id)
    if not distribs:
        return {
            "wrims_has_record": False,
            "wrims_invasive": None,
            "wrims_establishment": None,
            "wrims_matching_mrgid": None
        }

    # Build index: MRGID -> list of distribution entries
    mrgid_to_records = {}
    for d in distribs:
        loc_id = d.get("locationID")
        mrgid = extract_mrgid_from_locationID(loc_id)
        if not mrgid:
            continue
        mrgid_to_records.setdefault(mrgid, []).append(d)

    # Check if any of the location's MRGIDs appears in the distribution records
    for loc_mrgid in mrgids_for_location:
        recs = mrgid_to_records.get(loc_mrgid)
        if not recs:
            continue

        best_invasive = None
        best_establishment = None

        for d in recs:
            invasiveness = (d.get("invasiveness") or "").lower()
            establishment = (d.get("establishmentMeans") or "").lower()

            if "invasive" in invasiveness:
                best_invasive = "invasive"
                best_establishment = establishment or None
                break

            if any(word in establishment for word in
                   ["introduced", "alien", "non-indigenous", "nonindigenous"]):
                best_invasive = "introduced"
                best_establishment = establishment or "introduced"

        if best_invasive is not None:
            return {
                "wrims_has_record": True,
                "wrims_invasive": best_invasive,
                "wrims_establishment": best_establishment,
                "wrims_matching_mrgid": loc_mrgid,
            }

    # there are distributions, but nothing matching those MRGIDs with invasiveness info
    return {
        "wrims_has_record": True,
        "wrims_invasive": None,
        "wrims_establishment": None,
        "wrims_matching_mrgid": None
    }


# .....
def wrims_label(status_dict):
    """
    Human-friendly label.
    """
    if not status_dict["wrims_has_record"]:
        return "no WRiMS distribution record"

    inv = status_dict["wrims_invasive"]
    if inv == "invasive":
        return "probably invasive"
    if inv == "introduced":
        return "probably introduced"

    return "unknown status (possibly native)"


# start
# -----
# def classify_with_progress(scientific_name, dict_locations):

# finish
# -----
with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")

print(f"Finish: {workflow_step}")


workflow name
Finish: NIS-Classify with Progress


In [6]:
# Main Calculation
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - dummy_cell_arg_i: String
#   outputs:
#    - dummy_cell_arg_o: String
# ...

import os
import sys
from datetime import datetime

print(param_workflow_name)
workflow_step = f"{conf_vlab_name}-Main Calculation"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 

# lib, minio_public
# -----
# sys.path.append(conf_minio_public_local_code)

# lib, minio_user
# -----
# sys.path.append(conf_minio_user_local_code)

# input
# -----
dummy_cell_arg_i = "dummy input"

# output
# -----
dummy_cell_arg_o = "dummy output"

# func
# -----

# 1. OBIS helpers: occurrences + sea-only distance
def find_closest_obis_occurrence(scientific_name, new_lat, new_lon, limit=200):
    """
    From all OBIS occurrences of 'scientific_name', find the one with
    the shortest sea-only distance to (new_lat, new_lon).
    Shows a tqdm progress bar for the sea-routing.
    """
    occurrences = get_obis_occurrences(scientific_name, limit=limit)
    if not occurrences:
        raise RuntimeError(
            f"No OBIS occurrences with coordinates found for '{scientific_name}'."
        )

    best = None

    # Progress bar over all OBIS records
    for rec in tqdm(occurrences,
                    desc=f"Sea-routing OBIS for {scientific_name}",
                    unit="record",
                    leave=False):
        lat = rec["decimalLatitude"]
        lon = rec["decimalLongitude"]
        try:
            d = sea_distance_km(new_lat, new_lon, lat, lon)
        except Exception as e:
            print(f"[WARN] sea routing failed for ({lat}, {lon}): {e}")
            d = None

        rec["sea_distance_km"] = d
        if d is None:
            continue

        if best is None or d < best["sea_distance_km"]:
            best = rec

    if best is None:
        raise RuntimeError(
            f"No valid sea-only distances could be computed to OBIS occurrences "
            f"for '{scientific_name}'."
        )

    return best


# 3. WoRMS/WRiMS helpers
def classify_with_progress(scientific_name, dict_locations):
    """
    Run WRiMS classification for each (lat, lon) in `locations`
    with a tqdm progress bar.

    dict_locations: dict[label] = (lat, lon)

    Returns dict[label] = {
        "mrgids": [...],
        "status": {...},
        "label": "human-friendly label"
    }
    """
    results = {}

    for key in tqdm(dict_locations.keys(),
                    desc=f"WRiMS status for {scientific_name}",
                    unit="location",
                    leave=False):
        lat, lon = dict_locations[key]
        mrgids = get_filtered_mrgids_for_coord(
            lat, lon, max_diagonal_km=param_MAX_REGION_DIAGONAL_KM
        )
        status = classify_invasiveness(scientific_name, mrgids)
        results[key] = {
            "mrgids": mrgids,
            "status": status,
            "label": wrims_label(status)
        }

    return results


# start
# -----
# Main-4 Loop over each row and run your full analysis
# .....
for _, row in tqdm(df_merged.iterrows(),
                    total=len(df_merged),
                    desc="Total analyses",
                    unit="case"):
    loc_id     = row["locationID"]
    verb_lat   = row["verbatimLatitude"]
    verb_lon   = row["verbatimLongitude"]
    lat        = float(row["decimalLatitude"])
    lon        = float(row["decimalLongitude"])
    sci_name   = row["scientificName"]
    taxon_rank = row.get("taxonRank", "")

    # ---- OBIS: closest occurrence by sea-only distance ----
    try:
        closest = find_closest_obis_occurrence(
            sci_name,
            lat,
            lon,
            limit=param_MAX_OBIS_RECORDS
        )
        obis_lat     = closest["decimalLatitude"]
        obis_lon     = closest["decimalLongitude"]
        obis_dist_km = closest["sea_distance_km"]
    except Exception as e:
        print(f"[WARN] OBIS step failed for {sci_name} at {loc_id}: {e}")
        closest      = None
        obis_lat     = None
        obis_lon     = None
        obis_dist_km = None

    # ---- WRiMS classification for new & closest OBIS df_locations ----
    locations_dict = {
        "new_observation": (lat, lon)
    }
    if closest is not None:
        locations_dict["closest_obis"] = (obis_lat, obis_lon)

    wrims_results = classify_with_progress(sci_name, locations_dict)

    new_label  = wrims_results["new_observation"]["label"]
    obis_label = wrims_results.get("closest_obis", {}).get("label")

    # ---- Collect row for output ----
    results.append(
        {
            "locationID":                loc_id,
            "verbatimLatitude":          verb_lat,
            "verbatimLongitude":         verb_lon,
            "decimalLatitude":           lat,
            "decimalLongitude":          lon,
            "scientificName":            sci_name,
            "taxonRank":                 taxon_rank,
            "closest_obis_lat":          obis_lat,
            "closest_obis_lon":          obis_lon,
            "distance_over_water_km":    obis_dist_km,
            "wrims_status_new_location": new_label,
            "wrims_status_closest_obis": obis_label
        }
    )

# Main-5 Write single TSV output
# .....
out_df = pd.DataFrame(results)
out_df.to_csv(param_file_output, sep="\t", index=False)
print(f"\nDone. Results written to: {param_file_output}")

# finish
# -----
with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")
    fp_log.write(f"\nOutput: {param_file_output}\n")

print(f"Finish: {workflow_step}")


workflow name


Sea-routing OBIS for Acartia tonsa: 100%|█████████▉| 9995/10000 [00:37<00:00, 283.05record/s]
                                                                                             
Sea-routing OBIS for Ammothea hilgendorfi: 100%|█████████▉| 275/276 [00:02<00:00, 86.26record/s]
                                                                                                
Sea-routing OBIS for Amphibalanus amphitrite:  99%|█████████▉| 1355/1363 [00:18<00:00, 71.29record/s]
                                                                                                     
Sea-routing OBIS for Amphibalanus improvisus: 100%|█████████▉| 9980/10000 [00:18<00:00, 575.55record/s]
                                                                                                       
Sea-routing OBIS for Ampithoe valida:  99%|█████████▉| 538/541 [00:07<00:00, 71.93record/s]
                                                                                           
Sea-routing OBIS for A

[WARN] OBIS step failed for Cryptorchestia garbinii at Site A: No OBIS occurrences with coordinates found for 'Cryptorchestia garbinii'.



Sea-routing OBIS for Dasysiphonia japonica: 100%|█████████▉| 1756/1763 [00:08<00:00, 229.72record/s]
                                                                                                    
Sea-routing OBIS for Diadumene lineata:  99%|█████████▉| 960/966 [00:09<00:00, 107.12record/s]
                                                                                              
Sea-routing OBIS for Dikerogammarus villosus:   0%|          | 0/54 [00:00<?, ?record/s]
                                                                                        
Sea-routing OBIS for Diplosoma listerianum:  99%|█████████▉| 3085/3102 [00:12<00:00, 290.61record/s]
                                                                                                    
Sea-routing OBIS for Ensis leei:  99%|█████████▊| 9855/10000 [00:04<00:00, 1922.63record/s]
                                                                                           
Sea-routing OBIS for Eriocheir sinensis:  9


Done. Results written to: /home/jovyan/Cloud Storage/naa-vre-user-data/NIS/workflow name/output-results.tsv
Finish: NIS-Main Calculation
